## Data

### 1. Import libraries

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid", palette="deep")

print("Libraries loaded successfully.")

### 2. Load the CSV

In [ ]:
FILE_NAME = "Nuclear_Technology_Investment_Dataset.csv"
possible_paths = [
    Path("data") / FILE_NAME,
    Path(FILE_NAME),
]

file_path = next((path for path in possible_paths if path.exists()), None)

if file_path is None:
    try:
        from google.colab import files
        print(f"Please upload {FILE_NAME}")
        uploaded = files.upload()
        file_path = Path(FILE_NAME)
    except ImportError as error:
        raise FileNotFoundError(
            f"{FILE_NAME} was not found. Place it in the data folder or beside the notebook."
        ) from error

technology_data = pd.read_csv(file_path)
print(f"Loaded {technology_data.shape[0]} technologies and {technology_data.shape[1]} columns.")
display(technology_data.head())

### 3. Validate structure and data quality

In [ ]:
required_columns = [
    "Technology_ID",
    "Technology_Name",
    "Technology_Readiness_1_9",
    "Market_Potential_1_10",
    "Strategic_Fit_1_10",
    "Safety_Impact_1_10",
    "Partnership_Potential_1_10",
    "Financial_Attractiveness_1_10",
    "Regulatory_Complexity_1_10",
    "Estimated_CAPEX_CAD",
    "Annual_Operating_Cost_CAD",
    "Estimated_Annual_Revenue_CAD",
    "Estimated_Annual_Cost_Savings_CAD",
    "Weighted_Opportunity_Score_1_10",
]

missing_columns = [column for column in required_columns if column not in technology_data.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

quality_summary = pd.DataFrame({
    "Check": [
        "Rows",
        "Columns",
        "Missing values",
        "Duplicate rows",
        "Duplicate technology IDs",
    ],
    "Result": [
        technology_data.shape[0],
        technology_data.shape[1],
        int(technology_data.isna().sum().sum()),
        int(technology_data.duplicated().sum()),
        int(technology_data["Technology_ID"].duplicated().sum()),
    ],
})

display(quality_summary)

assert technology_data.isna().sum().sum() == 0, "The dataset contains missing values."
assert technology_data.duplicated().sum() == 0, "The dataset contains duplicate rows."
assert technology_data["Technology_ID"].is_unique, "Technology IDs must be unique."

print("Data-quality checks passed.")

### 4. Recalculate and verify the opportunity score

In [ ]:
score_weights = {
    "Technology_Readiness_1_9": 0.15,
    "Market_Potential_1_10": 0.20,
    "Strategic_Fit_1_10": 0.25,
    "Safety_Impact_1_10": 0.20,
    "Partnership_Potential_1_10": 0.10,
    "Financial_Attractiveness_1_10": 0.10
}

technology_data["Calculated_Opportunity_Score"] = sum(
    technology_data[column] * weight
    for column, weight in score_weights.items()
).round(2)

technology_data["Weighted_Opportunity_Score_1_10"] = (
    technology_data["Calculated_Opportunity_Score"]
)

print("Opportunity scores calculated successfully.")

display(
    technology_data[
        [
            "Technology_Name",
            "Calculated_Opportunity_Score"
        ]
    ].sort_values(
        "Calculated_Opportunity_Score",
        ascending=False
    )
)

## Results

### 5. Calculate business-case metrics

In [ ]:
technology_data["Total_Annual_Benefit_CAD"] = (
    technology_data["Estimated_Annual_Revenue_CAD"]
    + technology_data["Estimated_Annual_Cost_Savings_CAD"]
)

technology_data["Net_Annual_Benefit_CAD"] = (
    technology_data["Total_Annual_Benefit_CAD"]
    - technology_data["Annual_Operating_Cost_CAD"]
)

technology_data["First_Year_ROI_Pct"] = np.where(
    technology_data["Estimated_CAPEX_CAD"] > 0,
    technology_data["Net_Annual_Benefit_CAD"]
    / technology_data["Estimated_CAPEX_CAD"] * 100,
    np.nan,
)

technology_data["Payback_Period_Years"] = np.where(
    technology_data["Net_Annual_Benefit_CAD"] > 0,
    technology_data["Estimated_CAPEX_CAD"]
    / technology_data["Net_Annual_Benefit_CAD"],
    np.nan,
)

technology_data["Three_Year_Net_Value_CAD"] = (
    technology_data["Net_Annual_Benefit_CAD"] * 3
    - technology_data["Estimated_CAPEX_CAD"]
)

technology_data["Three_Year_ROI_Pct"] = np.where(
    technology_data["Estimated_CAPEX_CAD"] > 0,
    technology_data["Three_Year_Net_Value_CAD"]
    / technology_data["Estimated_CAPEX_CAD"] * 100,
    np.nan,
)

financial_view = technology_data[
    [
        "Technology_Name",
        "Estimated_CAPEX_CAD",
        "Net_Annual_Benefit_CAD",
        "First_Year_ROI_Pct",
        "Payback_Period_Years",
        "Three_Year_ROI_Pct",
    ]
].sort_values("Three_Year_ROI_Pct", ascending=False)

display(financial_view.round(2))

### 6. Executive summary

In [ ]:
top_opportunity = technology_data.loc[
    technology_data["Calculated_Opportunity_Score"].idxmax()
]
fastest_payback = technology_data.loc[
    technology_data["Payback_Period_Years"].idxmin()
]
short_term_candidates = technology_data[
    (technology_data["Development_Months"] <= 18)
    & (technology_data["Calculated_Opportunity_Score"] >= 8.50)
].sort_values("Calculated_Opportunity_Score", ascending=False)

print("EXECUTIVE SUMMARY")
print("-" * 70)
print(
    f"Highest opportunity score: {top_opportunity['Technology_Name']} "
    f"({top_opportunity['Calculated_Opportunity_Score']:.2f}/10)"
)
print(
    f"Fastest estimated payback: {fastest_payback['Technology_Name']} "
    f"({fastest_payback['Payback_Period_Years']:.2f} years)"
)
print(f"Short-term pilot candidates: {len(short_term_candidates)}")
print(
    "Candidate names: "
    + ", ".join(short_term_candidates["Technology_Name"].tolist())
)
print()
print("Reminder: Results are based on hypothetical demonstration assumptions.")

### 7. Compare opportunity scores

In [ ]:
score_plot_data = technology_data.sort_values(
    "Calculated_Opportunity_Score",
    ascending=True,
)

plt.figure(figsize=(11, 6))
bars = plt.barh(
    score_plot_data["Technology_Name"],
    score_plot_data["Calculated_Opportunity_Score"],
    color="#2563EB",
)
plt.xlim(0, 10)
plt.xlabel("Weighted opportunity score (out of 10)")
plt.ylabel("")
plt.title("Weighted Opportunity Score by Technology")

for bar, score in zip(bars, score_plot_data["Calculated_Opportunity_Score"]):
    plt.text(
        score + 0.08,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.2f}",
        va="center",
    )

plt.tight_layout()
plt.show()

### 8. Compare CAPEX and annual benefit

In [ ]:
plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=technology_data,
    x="Estimated_CAPEX_CAD",
    y="Net_Annual_Benefit_CAD",
    size="Calculated_Opportunity_Score",
    hue="Risk_Level",
    sizes=(120, 420),
    alpha=0.85,
)

for _, row in technology_data.iterrows():
    plt.annotate(
        row["Technology_ID"],
        (row["Estimated_CAPEX_CAD"], row["Net_Annual_Benefit_CAD"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
    )

plt.xlabel("Estimated CAPEX (CAD)")
plt.ylabel("Estimated net annual benefit (CAD)")
plt.title("Estimated CAPEX vs. Net Annual Benefit")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display(
    technology_data[
        ["Technology_ID", "Technology_Name", "Risk_Level"]
    ].sort_values("Technology_ID")
)

### 9. Compare estimated payback periods

In [ ]:
payback_plot_data = technology_data.sort_values(
    "Payback_Period_Years",
    ascending=True,
)

plt.figure(figsize=(11, 6))
sns.barplot(
    data=payback_plot_data,
    x="Payback_Period_Years",
    y="Technology_Name",
    color="#0F766E",
)
plt.xlabel("Estimated payback period (years)")
plt.ylabel("")
plt.title("Estimated Payback Period by Technology")
plt.tight_layout()
plt.show()

### 10. Scenario analysis for the highest-scoring opportunity

In [ ]:
selected_technology = top_opportunity.copy()

scenario_inputs = pd.DataFrame({
    "Scenario": ["Conservative", "Expected", "Optimistic"],
    "Benefit_Multiplier": [0.75, 1.00, 1.25],
    "Operating_Cost_Multiplier": [1.10, 1.00, 0.95],
})

scenario_inputs["Adjusted_Annual_Benefit_CAD"] = (
    selected_technology["Total_Annual_Benefit_CAD"]
    * scenario_inputs["Benefit_Multiplier"]
)
scenario_inputs["Adjusted_Operating_Cost_CAD"] = (
    selected_technology["Annual_Operating_Cost_CAD"]
    * scenario_inputs["Operating_Cost_Multiplier"]
)
scenario_inputs["Adjusted_Net_Annual_Benefit_CAD"] = (
    scenario_inputs["Adjusted_Annual_Benefit_CAD"]
    - scenario_inputs["Adjusted_Operating_Cost_CAD"]
)
scenario_inputs["Payback_Period_Years"] = (
    selected_technology["Estimated_CAPEX_CAD"]
    / scenario_inputs["Adjusted_Net_Annual_Benefit_CAD"]
)
scenario_inputs["Three_Year_ROI_Pct"] = (
    (
        scenario_inputs["Adjusted_Net_Annual_Benefit_CAD"] * 3
        - selected_technology["Estimated_CAPEX_CAD"]
    )
    / selected_technology["Estimated_CAPEX_CAD"]
    * 100
)

print(f"Scenario analysis: {selected_technology['Technology_Name']}")
display(scenario_inputs.round(2))

## Takeaways

1. The opportunity score and financial model measure different dimensions. A high strategic score does not automatically mean the lowest cost or fastest payback.
2. Shorter-term technologies can be moved into controlled proofs of concept, while higher-complexity opportunities may be better suited to partnerships and longer roadmaps.
3. Regulatory complexity, nuclear quality assurance, cybersecurity, safety validation, expert review, and customer evidence must be evaluated before any real investment decision.
4. The model is designed to support structured discussion, not replace engineering or executive judgment.

### 11. Export the Power BI dataset

In [ ]:
POWER_BI_FILE = "Nuclear_Technology_PowerBI_Ready.csv"

export_data = technology_data.drop(
    columns=["Original_Stored_Score", "Score_Difference"],
    errors="ignore",
).copy()
export_data.to_csv(POWER_BI_FILE, index=False)

print(f"Power BI file created: {POWER_BI_FILE}")
print(f"Exported rows: {len(export_data)}")
display(export_data.head())

try:
    from google.colab import files
    files.download(POWER_BI_FILE)
except ImportError:
    print(f"File saved locally as {POWER_BI_FILE}")